In [1]:
import os
import json
import base64
import requests
import pandas as pd
from datetime import date, timedelta
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.environ["AMPLITUDE_API_KEY"]
API_SECRET = os.environ["AMPLITUDE_API_SECRET"]

_token = base64.b64encode(f"{API_KEY}:{API_SECRET}".encode()).decode()
AUTH_HEADER = {"Authorization": f"Basic {_token}"}
ENDPOINT = "https://amplitude.com/api/2/events/segmentation"

print("Credenciales cargadas. Key:", API_KEY[:6] + "...")


Credenciales cargadas. Key: eab2de...


In [ ]:
# --- Parseo modo parseLong: una fila por (combinacion x dia) ---

def parse_amplitude_response(resp_json, group_by_fields):
    """
    Convierte la respuesta de /events/segmentation en filas planas.
    - series[i][d] = valor para la combinacion i en el dia xValues[d]
    - seriesLabels[i] = label "v1, v2, v3, ..." con los valores del group_by en orden
    Devuelve list[dict] con {date, <campos group_by>, totals} solo cuando value > 0.
    Las fechas se devuelven TAL CUAL vienen de Amplitude (TZ Santiago, sin convertir).
    """
    data = resp_json.get("data", {})
    series = data.get("series", [])
    labels = data.get("seriesLabels", [])
    x_values = data.get("xValues", [])

    rows = []
    n_cols = len(group_by_fields)

    for i, serie in enumerate(series):
        label = labels[i] if i < len(labels) else ""

        # El doc menciona que algunos labels usan "; " en vez de ", ".
        parts = label.split(", ")
        if len(parts) < n_cols and "; " in label:
            parts = label.split("; ")

        # Si faltan partes, rellenar con "(none)" (mismo comportamiento que el pipeline TS)
        if len(parts) < n_cols:
            parts = parts + ["(none)"] * (n_cols - len(parts))

        label_map = {field: parts[k] for k, field in enumerate(group_by_fields)}

        for d, value in enumerate(serie):
            if value and value > 0:
                row = {"date": x_values[d] if d < len(x_values) else None}
                row.update(label_map)
                row["totals"] = value
                rows.append(row)

    return rows


In [ ]:
# --- Config editable: cambia filtros, ventana o group_by aqui ---

fecha_corte = date.today()
dias_ventana = 12
dias_bucket = 3   # 12 / 3 = 4 buckets

event_config = {
    "event_type": "Product Updated",
    "filters": [
        {
            "subprop_type": "event",
            "subprop_key": "country_code",
            "subprop_op": "is",
            "subprop_value": ["MX"],
        },
        {
            "subprop_type": "event",
            "subprop_key": "sales_channel",
            "subprop_op": "is",
            "subprop_value": ["dealer"],
        },
    ],
    "group_by": [
        {"type": "event", "value": "lead_id"},
        {"type": "event", "value": "brand"},
        {"type": "event", "value": "model"},
        {"type": "event", "value": "color"},
        {"type": "event", "value": "price_net"},
        {"type": "event", "value": "year"},
        {"type": "event", "value": "store_name"},
        {"type": "event", "value": "store_alias"},
    ],
}

# Lista de nombres de columnas que saldra del parser, en el mismo orden del group_by
group_by_fields = [g["value"] for g in event_config["group_by"]]

print("fecha_corte:", fecha_corte)
print("ventana:", dias_ventana, "dias =", dias_ventana // dias_bucket, "buckets de", dias_bucket, "dias")
print("group_by:", group_by_fields)


In [ ]:
# --- Extraccion: 4 buckets contiguos -> parser -> DataFrame ---

n_buckets = dias_ventana // dias_bucket

# Generamos los buckets de mas viejo a mas nuevo, terminando en fecha_corte.
# Cada bucket cubre dias_bucket dias inclusive (ej: -11 -> -9, -8 -> -6, ...).
buckets = []
for b in range(n_buckets):
    end_offset = (n_buckets - 1 - b) * dias_bucket
    start_offset = end_offset + (dias_bucket - 1)
    bucket_start = fecha_corte - timedelta(days=start_offset)
    bucket_end = fecha_corte - timedelta(days=end_offset)
    buckets.append((bucket_start, bucket_end))

all_rows = []

for bstart, bend in buckets:
    params = {
        "e": json.dumps(event_config),
        "s": "[]",
        "start": bstart.strftime("%Y%m%d"),
        "end": bend.strftime("%Y%m%d"),
        "m": "totals",
        "i": 1,
        "limit": 10000,
    }

    resp = requests.get(ENDPOINT, params=params, headers=AUTH_HEADER, timeout=120)
    resp.raise_for_status()
    payload = resp.json()

    n_series = len(payload.get("data", {}).get("series", []))
    rows = parse_amplitude_response(payload, group_by_fields)

    warn = "  <-- WARNING: cerca del limite de 10k, posible truncamiento" if n_series >= 9000 else ""
    print(f"Bucket {bstart} -> {bend}: {n_series} series, {len(rows)} filas{warn}")

    all_rows.extend(rows)

df = pd.DataFrame(all_rows)
print("\nTotal filas:", len(df))
print("lead_id unicos:", df["lead_id"].nunique() if "lead_id" in df.columns else "n/a")
df.head()
